In [1]:
import sys
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import glob
import plotly.graph_objects as go
import os
import numpy as np
import at3d
import matplotlib.pyplot as plt
import xarray as xr
from util import load_sensor_dict
from scipy.interpolate import griddata
from collections import defaultdict

In [2]:
def interp_sensor_to_rte_grid(sensor, rte_grid, interp_var_name):
    x_sensor = sensor.ray_x.data
    y_sensor = sensor.ray_y.data
    interp_var = sensor[interp_var_name].data

    x_interp_to = rte_grid.x.data
    y_interp_to = rte_grid.y.data

    grid_x, grid_y = np.meshgrid(x_interp_to, y_interp_to, indexing='ij')
    var_interpolated = griddata((x_sensor, y_sensor), interp_var, (grid_x, grid_y), method='linear')
    rte_grid[interp_var_name] = (['x', 'y'], var_interpolated)
    return rte_grid

In [4]:
def load_retreival(dir, cm_names):
    assert os.path.isdir(dir + '/iterations/')
    name = os.path.split(dir)[-1]
    files = np.sort(glob.glob(dir + '/iterations/iter*.nc'))
    sensors, solvers, rte_grid = at3d.util.load_forward_model(files[-1])
    cloud = solvers[0.67].medium['cloud']
    cloud['tau'] = cloud.extinction.integrate('z')

    tau1D_file = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/AT3D_1DTAU_CAMP2Ex_RF17_default_1400.nc'
    obs = load_sensor_dict(tau1D_file)
    tau_1D_sensor = obs['MISR']['sensor_list'][4]
    cloud_1D = rte_grid.copy(deep=True)
    cloud_1D.attrs = tau_1D_sensor.attrs
    cloud_1D = interp_sensor_to_rte_grid(tau_1D_sensor, cloud_1D, 'tau')
    cloud_1D = interp_sensor_to_rte_grid(tau_1D_sensor, cloud_1D, 'I')

    cvm_file = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/CVM_default_open.nc'
    cvm = xr.load_dataset(cvm_file)
    cvm['cloud_mask_cvm'] = (cvm.cloud_volume_mask.integrate('z') > 0).astype(int)
    cvm['cloud_mask_at3d'] = (cloud.tau > 0).astype(int)
    th_mask = (((np.pi * cloud_1D.I) / cloud_1D.mu0) > 0.04).astype(int).data
    th_mask[0,:] = th_mask[-1,:] = th_mask[:,0] = th_mask[:,-1] = 0
    cvm['cloud_mask_th'] = (['x', 'y'], th_mask)

    data = {}
    for cm in cm_names:
        data['tau1d_{}'.format(cm)] = cloud_1D.tau.where(cvm[cm] > 0).mean().data
        data['tau3d_{}'.format(cm)] = cloud.tau.where(cvm[cm] > 0).mean().data

    return data

In [5]:
def traverse(dir, data, cm_names, parent_key=()):
    name = os.path.split(dir)[-1]
    parent_key += (name,)
    if(os.path.isdir(dir + '/iterations/')):
        data[parent_key] = load_retreival(dir, cm_names)
    else:
        for n in os.listdir(dir):
            traverse(dir+'/'+n, data, cm_names, parent_key)


In [6]:
version_dir = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/AT3D_output/V03'
data = defaultdict(defaultdict)
cm_names = ['cloud_mask_cvm', 'cloud_mask_at3d', 'cloud_mask_th']
traverse(version_dir, data, cm_names)

/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '68'. Max cell optical depth: '3.3179593086242676'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3

In [27]:
print('{:<48} | {:<7} | {:<7} | {:<8} | {:<8} | {:<8} | {:<8} | {:<7} | {:<7} | {:<8} '.format(
    'Path',
    '1D: CVM',
    '3D: CVM',
    'increase',
    '1D: at3d',
    '3D: at3d',
    'increase',
    '1D: TH',
    '3D: TH',
    'increase'
))
print('-'*145)
for k, v in data.items():
    print('{:<48} | {:<7.3f} | {:<7.3f} | {:<8.2f} | {:<8.3f} | {:<8.3f} | {:<8.2f} | {:<7.3f} | {:<7.3f} | {:<8.2f}'.format(
        str(k),
        v['tau1d_cloud_mask_cvm'],
        v['tau3d_cloud_mask_cvm'],
        100*(v['tau3d_cloud_mask_cvm'] - v['tau1d_cloud_mask_cvm']) / v['tau3d_cloud_mask_cvm'],
        v['tau1d_cloud_mask_at3d'],
        v['tau3d_cloud_mask_at3d'],
        100*(v['tau3d_cloud_mask_at3d'] - v['tau1d_cloud_mask_at3d']) / v['tau3d_cloud_mask_at3d'],
        v['tau1d_cloud_mask_th'],
        v['tau3d_cloud_mask_th'],
        100*(v['tau3d_cloud_mask_th'] - v['tau1d_cloud_mask_th']) / v['tau3d_cloud_mask_th'],
    ))

Path                                             | 1D: CVM | 3D: CVM | increase | 1D: at3d | 3D: at3d | increase | 1D: TH  | 3D: TH  | increase 
-------------------------------------------------------------------------------------------------------------------------------------------------
('V03', 'default')                               | 0.492   | 0.560   | 12.16    | 0.688    | 0.804    | 14.49    | 0.595   | 0.676   | 11.99   
('V03', 'aerosol', '0p150')                      | 0.492   | 0.693   | 28.98    | 0.494    | 0.695    | 29.01    | 0.595   | 0.818   | 27.19   
('V03', 'aerosol', '0p250')                      | 0.492   | 0.483   | -1.84    | 1.081    | 1.204    | 10.22    | 0.595   | 0.584   | -1.87   
('V03', 'aerosol', 'mix22')                      | 0.492   | 0.577   | 14.65    | 0.653    | 0.780    | 16.29    | 0.595   | 0.696   | 14.47   
('V03', 'aerosol', 'mix54')                      | 0.492   | 0.530   | 7.05     | 0.771    | 0.869    | 11.27    | 0.595   | 0.640   